<a href="https://colab.research.google.com/github/meganandrus1122-rgb/universal2dto3dmodelpipelineusingcolab/blob/main/Universal_2D_to_3D_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Orca 2D-to-3D Reconstruction Pipeline
This notebook synthesizes the raw 3D mesh (`orca_raw.obj`) directly from your saved 3/4 perspective image in Google Drive using Hunyuan3D-2.

In [ ]:
# ==============================================================================
# Cell 1: Mount Google Drive & Environment Setup
# ==============================================================================
from google.colab import drive
import os
import sys

# Mount Google Drive to access your project files
try:
    drive.mount('/content/drive', force_remount=True)
except Exception as e:
    print(f"Drive mount failed: {e}")

# Define project directory
project_dir = "/content/drive/MyDrive/3D Print Projects"
os.makedirs(project_dir, exist_ok=True)

# Clone the reconstruction pipeline if it doesn't exist
if not os.path.exists('/content/Hunyuan3D-2'):
    !git clone --recurse-submodules https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git /content/Hunyuan3D-2

%cd /content/Hunyuan3D-2

# Upgrade pip first
!pip install -q --upgrade pip

# Install pre-compiled compatible versions of tokenizers and transformers for Python 3.13
!pip install -q tokenizers==0.21.0 transformers==4.48.2 --prefer-binary

# Install requirements (excluding conflicting packages already handled above)
!pip install -q -r requirements.txt

# Optimized requirements to keep pipeline fully compatible
!pip install -q "jedi>=0.16" "numpy<2.5" "cuda-bindings~=13.0.3" "numba<=0.65.1,>=0.58" "rich<14" "fsspec[http]<=2026.2.0,>=2023.1.0" "huggingface_hub<2.0" trimesh accelerate diffusers datasets torchvision "pillow>=12.1.0,<14.0.0"

# Install the repository as an editable package
!pip install -q -e .

print("Environment setup complete! Please restart the session using the cell below if imports fail.")

Mounted at /content/drive
/content/Hunyuan3D-2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ...

In [ ]:
import os
# [Diagnostic Cell] Force a complete runtime session restart to apply the new package installations.
# Only run this once after initial setup if you encounter module import errors.
os.kill(os.getpid(), 9)

In [ ]:
import PIL
print(f"Loaded Pillow version: {PIL.__version__}")

Loaded Pillow version: 12.3.0


In [ ]:
import os

# [Diagnostic Cell] Check the contents of the /content directory to verify repository cloning
print("Root /content directory contents:")
print(os.listdir('/content'))

possible_path_1 = '/content/Hunyuan3D-2'
print(f"\nDoes {possible_path_1} exist?: {os.path.exists(possible_path_1)}")
if os.path.exists(possible_path_1):
    print(f"Contents of {possible_path_1}: {os.listdir(possible_path_1)[:5]}")

Root /content directory contents:
['.config', 'drive', 'Hunyuan3D-2', 'Gemini_Generated_Image_vy9wrqvy9wrqvy9w.jpg', 'sample_data']

Does /content/Hunyuan3D-2 exist?: True
Contents of /content/Hunyuan3D-2: ['.gitignore', 'NOTICE', 'api_server.py', 'hy3dgen', 'README_zh_cn.md']


In [ ]:
# @title 🎯 3D Synthesis Configuration & Reusable Workflow { run: "auto" }

import os
import sys
import PIL.Image
import torch

# --------------------------------------------------
# REUSABLE WORKFLOW DEFINITION
# --------------------------------------------------
def run_hunyuan3d_workflow(
    input_filename="input_model.png",
    output_filename="output_mesh.obj",
    project_folder_name="3D Print Projects",
    drive_base="/content/drive/MyDrive"
):
    """
    Runs the Hunyuan3D-2 2D-to-3D synthesis pipeline.

    Parameters:
      input_filename (str): Name of the input 3/4 perspective image in Google Drive.
      output_filename (str): Name of the output raw 3D mesh file (e.g., .obj, .gltf).
      project_folder_name (str): Google Drive subfolder for the project.
      drive_base (str): Path to the mounted Google Drive.
    """
    # Path resolution
    project_dir = os.path.join(drive_base, project_folder_name)
    input_image_path = os.path.join(project_dir, input_filename)
    output_mesh_path = os.path.join(project_dir, output_filename)

    # Ensure project output directory exists
    os.makedirs(project_dir, exist_ok=True)

    # Monkey-patch ANTIALIAS for modern Pillow compatibility
    if not hasattr(PIL.Image, "ANTIALIAS"):
        PIL.Image.ANTIALIAS = PIL.Image.LANCZOS

    # Ensure repo module is importable
    repo_path = "/content/Hunyuan3D-2"
    if repo_path not in sys.path:
        sys.path.insert(0, repo_path)

    from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline

    # Validation
    if not os.path.exists(input_image_path):
        raise FileNotFoundError(f"Input image not found: {input_image_path}")

    print(f"Loading input image: {input_image_path}")
    print("Initializing Hunyuan3D Pipeline on GPU...")

    # Load pipeline (weights stay cached if already downloaded)
    pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
        "tencent/Hunyuan3D-2", torch_dtype=torch.float16, device="cuda"
    )

    print(f"Synthesizing 3D mesh for ‘{output_filename}’...")
    mesh = pipeline(image=input_image_path)[0]
    mesh.export(output_mesh_path)

    print(f"\n✅ Success! Mesh saved to: {output_mesh_path}")
    print(
        f"File size: {os.path.getsize(output_mesh_path) / (1024 * 1024):.2f} MB"
    )
    return output_mesh_path

In [ ]:
import os

project_dir = "/content/drive/MyDrive/3D Print Projects"
output_mesh = os.path.join(project_dir, "orca_raw.obj")

if os.path.exists(output_mesh):
    size_bytes = os.path.getsize(output_mesh)
    size_mb = size_bytes / (1024 * 1024)
    print(f"✅ Success! The 3D mesh is present in your Google Drive.")
    print(f"Location: {output_mesh}")
    print(f"File size: {size_mb:.2f} MB")
else:
    print(f"❌ File not found at: {output_mesh}")
    print("Please verify if the synthesis cell above has completed executing.")

❌ File not found at: /content/drive/MyDrive/3D Print Projects/orca_raw.obj
Please verify if the synthesis cell above has completed executing.


In [ ]:
# This cell is no longer needed. The code has been merged into the cell above.

# @title 🔍 Inspect Generated Mesh Properties & Quick Verification

import os
import trimesh

# Quick invocation example using the reusable workflow
try:
    # Run synthesis using default arguments (or custom parameters)
    output_mesh_path = run_hunyuan3d_workflow(
        input_filename="orca_3quarter_view.png",
        output_filename="orca_raw.obj",
        project_folder_name="3D Print Projects"
    )
    
    if os.path.exists(output_mesh_path):
        loaded_mesh = trimesh.load(output_mesh_path)

        print("=== MESH AUDIT ===")
        print(f"Vertices: {len(loaded_mesh.vertices):,}")
        print(f"Faces:    {len(loaded_mesh.faces):,}")
        print(f"Is Watertight (Manifold): {loaded_mesh.is_watertight}")
        print(
            f"Bounding Box Dimensions (XYZ): {loaded_mesh.extents.round(2)}"
        )
    else:
        print("Output mesh not found. Please verify the workflow run.")
except NameError:
    print("Workflow function 'run_hunyuan3d_workflow' is not defined. Please run cell 'IDuAzd4ZmCma' first.")
except Exception as e:
    print(f"An error occurred during workflow execution: {e}")

In [ ]:
# @title 🚀 3D Model Generator & STL Exporter Dashboard {display-mode: "form"}

import os
import json
import trimesh
from IPython.display import HTML, display
from google.colab import output

# Register error tracking callback
def _report_js_error(message):
    print(f"JavaScript Error: {message}")

output.register_callback('report_js_error', _report_js_error)

# Python callback function to trigger the 3D generation & conversion
def run_generation_and_convert(input_file, output_file, project_folder):
    try:
        # 1. Run the existing synthesis workflow (defined in previous cells)
        if 'run_hunyuan3d_workflow' not in globals():
            return json.dumps({"status": "error", "message": "Synthesis function 'run_hunyuan3d_workflow' is not defined. Please run the setup cells above."})

        # Resolve the actual path. If an absolute path is provided, use it directly.
        if os.path.isabs(input_file):
            input_image_path = input_file
        else:
            input_image_path = os.path.join("/content/drive/MyDrive", project_folder, input_file)

        # Define raw output mesh path in project folder
        obj_mesh_path = os.path.join("/content/drive/MyDrive", project_folder, output_file)

        # Force run workflow with resolved pathing
        import PIL.Image
        import torch
        import sys

        os.makedirs(os.path.dirname(obj_mesh_path), exist_ok=True)
        if not hasattr(PIL.Image, "ANTIALIAS"):
            PIL.Image.ANTIALIAS = PIL.Image.LANCZOS

        repo_path = "/content/Hunyuan3D-2"
        if repo_path not in sys.path:
            sys.path.insert(0, repo_path)

        from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline

        if not os.path.exists(input_image_path):
            return json.dumps({"status": "error", "message": f"Input image not found: {input_image_path}"})

        print(f"Loading input image: {input_image_path}")
        pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
            "tencent/Hunyuan3D-2", torch_dtype=torch.float16, device="cuda"
        )
        print(f"Synthesizing 3D mesh for ‘{output_file}’...")
        mesh = pipeline(image=input_image_path)[0]
        mesh.export(obj_mesh_path)

        if not os.path.exists(obj_mesh_path):
            return json.dumps({"status": "error", "message": "Failed to generate intermediate OBJ mesh file."})

        # 2. Convert OBJ to STL using trimesh
        loaded_mesh = trimesh.load(obj_mesh_path)

        # Define STL output path
        base_name, _ = os.path.splitext(obj_mesh_path)
        stl_mesh_path = base_name + ".stl"

        # Export as STL
        loaded_mesh.export(stl_mesh_path)

        # Gather mesh statistics for the dashboard indicators
        stats = {
            "status": "success",
            "vertices": len(loaded_mesh.vertices),
            "faces": len(loaded_mesh.faces),
            "watertight": loaded_mesh.is_watertight,
            "dimensions": [round(x, 2) for x in loaded_mesh.extents.tolist()],
            "obj_path": obj_mesh_path,
            "stl_path": stl_mesh_path,
            "file_size_mb": round(os.path.getsize(stl_mesh_path) / (1024 * 1024), 2)
        }
        return json.dumps(stats)
    except Exception as e:
        return json.dumps({"status": "error", "message": str(e)})

output.register_callback('run_generation_and_convert', run_generation_and_convert)

# Inline HTML/CSS/JS for the dashboard UI
html_content = """
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <style>
        body {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            background-color: #f4f6f8;
            color: #333;
            margin: 0;
            padding: 20px;
        }
        .dashboard-container {
            max-width: 1000px;
            margin: 0 auto;
            background: #ffffff;
            border-radius: 12px;
            box-shadow: 0 4px 15px rgba(0,0,0,0.05);
            padding: 30px;
        }
        .header {
            border-bottom: 2px solid #eaedf1;
            padding-bottom: 15px;
            margin-bottom: 25px;
            display: flex;
            align-items: center;
            justify-content: space-between;
        }
        .header h1 {
            margin: 0;
            font-size: 24px;
            color: #1a202c;
            display: flex;
            align-items: center;
            gap: 10px;
        }
        .row {
            display: flex;
            gap: 20px;
            margin-bottom: 25px;
            flex-wrap: wrap;
        }
        .col {
            flex: 1;
            min-width: 280px;
        }
        .card {
            background: #fdfdfd;
            border: 1px solid #e2e8f0;
            border-radius: 8px;
            padding: 20px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.02);
        }
        .card h3 {
            margin-top: 0;
            color: #4a5568;
            font-size: 16px;
            border-bottom: 1px solid #edf2f7;
            padding-bottom: 8px;
        }
        .form-group {
            margin-bottom: 15px;
        }
        .form-group label {
            display: block;
            margin-bottom: 6px;
            font-weight: 600;
            color: #4a5568;
            font-size: 14px;
        }
        .form-group input {
            width: 100%;
            padding: 10px;
            border: 1px solid #cbd5e0;
            border-radius: 6px;
            box-sizing: border-box;
            font-size: 14px;
            transition: border-color 0.2s;
        }
        .form-group input:focus {
            border-color: #3182ce;
            outline: none;
        }
        .btn {
            background-color: #3182ce;
            color: white;
            border: none;
            padding: 12px 20px;
            font-size: 15px;
            font-weight: bold;
            border-radius: 6px;
            cursor: pointer;
            width: 100%;
            transition: background-color 0.2s;
        }
        .btn:hover {
            background-color: #2b6cb0;
        }
        .btn:disabled {
            background-color: #cbd5e0;
            cursor: not-allowed;
        }
        .kpi-container {
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(180px, 1fr));
            gap: 15px;
            margin-bottom: 25px;
        }
        .kpi-card {
            background: #ebf8ff;
            border-left: 4px solid #3182ce;
            padding: 15px;
            border-radius: 6px;
            text-align: center;
        }
        .kpi-val {
            font-size: 20px;
            font-weight: bold;
            color: #2b6cb0;
            margin-top: 5px;
        }
        .kpi-label {
            font-size: 12px;
            color: #4a5568;
            text-transform: uppercase;
            letter-spacing: 0.5px;
        }
        #status-box {
            padding: 15px;
            border-radius: 6px;
            margin-top: 15px;
            font-weight: 500;
            display: none;
        }
        .status-success {
            background-color: #c6f6d5;
            color: #22543d;
            border: 1px solid #9ae6b4;
        }
        .status-error {
            background-color: #fed7d7;
            color: #742a2a;
            border: 1px solid #feb2b2;
        }
        .status-loading {
            background-color: #feebc8;
            color: #7b341e;
            border: 1px solid #fbd38d;
        }
    </style>
</head>
<body>

<div class="dashboard-container">
    <div class="header">
        <h1>🛠️ 3D Print Prep Dashboard (STL Exporter)</h1>
    </div>

    <div class="row">
        <div class="col">
            <div class="card">
                <h3>⚙️ Synthesis Configuration</h3>
                <div class="form-group">
                    <label for="project_folder">Project Subfolder (in Drive)</label>
                    <input type="text" id="project_folder" value="3D Print Projects">
                </div>
                <div class="form-group">
                    <label for="input_file">Input Image Path</label>
                    <input type="text" id="input_file" value="/content/Gemini_Generated_Image_vy9wrqvy9wrqvy9w.jpg">
                </div>
                <div class="form-group">
                    <label for="output_file">Output OBJ Filename</label>
                    <input type="text" id="output_file" value="custom_generation_raw.obj">
                </div>
                <button class="btn" id="generate-btn" onclick="triggerWorkflow()">Generate Model & Export STL</button>
                <div id="status-box"></div>
            </div>
        </div>

        <!-- KPI Stats / Audit Card -->
        <div class="col">
            <div class="card">
                <h3>📊 Model Audit Statistics</h3>
                <div class="kpi-container">
                    <div class="kpi-card">
                        <div class="kpi-label">Vertices</div>
                        <div class="kpi-val" id="val-vertices">-</div>
                    </div>
                    <div class="kpi-card">
                        <div class="kpi-label">Faces</div>
                        <div class="kpi-val" id="val-faces">-</div>
                    </div>
                    <div class="kpi-card">
                        <div class="kpi-label">Print Manifold</div>
                        <div class="kpi-val" id="val-watertight">-</div>
                    </div>
                    <div class="kpi-card">
                        <div class="kpi-label">STL File Size</div>
                        <div class="kpi-val" id="val-size">-</div>
                    </div>
                </div>

                <div class="form-group" style="margin-top: 15px;">
                    <label>Generated STL File Path</label>
                    <input type="text" id="stl-path-output" readonly placeholder="STL output path will show here...">
                </div>
            </div>
        </div>
    </div>
</div>

<script>
    window.onerror = function(message) {
        google.colab.kernel.invokeFunction('report_js_error', [message], {});
    };

    function setStatus(text, type) {
        const box = document.getElementById('status-box');
        box.style.display = 'block';
        box.className = '';
        if (type === 'success') box.classList.add('status-success');
        if (type === 'error') box.classList.add('status-error');
        if (type === 'loading') box.classList.add('status-loading');
        box.innerText = text;
    }

    function triggerWorkflow() {
        const btn = document.getElementById('generate-btn');
        const input_file = document.getElementById('input_file').value;
        const output_file = document.getElementById('output_file').value;
        const project_folder = document.getElementById('project_folder').value;

        btn.disabled = true;
        setStatus('Running 3D Synthesis & STL Conversion Pipeline... Please check your main execution terminal for details.', 'loading');

        google.colab.kernel.invokeFunction('run_generation_and_convert', [input_file, output_file, project_folder], {})
            .then(response => {
                btn.disabled = false;
                const data = JSON.parse(response.data['text/plain']);

                if (data.status === 'success') {
                    setStatus('✅ Mesh synthesis complete! STL successfully saved to Google Drive.', 'success');
                    document.getElementById('val-vertices').innerText = data.vertices.toLocaleString();
                    document.getElementById('val-faces').innerText = data.faces.toLocaleString();
                    document.getElementById('val-watertight').innerText = data.watertight ? 'Yes (Solid)' : 'No (Requires Slicer Repair)';
                    document.getElementById('val-size').innerText = data.file_size_mb + ' MB';
                    document.getElementById('stl-path-output').value = data.stl_path;
                } else {
                    setStatus('❌ Error: ' + data.message, 'error');
                }
            }).catch(err => {
                btn.disabled = false;
                setStatus('❌ Pipeline invocation failed: ' + err, 'error');
            });
    }
</script>
</body>
</html>
"""

display(HTML(html_content))


2026-11-03 05:45:20,650 - hy3dgen.shapgen - INFO - Try to load model from local path: /root/.cache/hy3dgen/tencent/Hunyuan3D-2/hunyuan3d-dit-v2-0
INFO:hy3dgen.shapgen:Try to load model from local path: /root/.cache/hy3dgen/tencent/Hunyuan3D-2/hunyuan3d-dit-v2-0
2026-11-03 05:45:20,651 - hy3dgen.shapgen - INFO - Model path not exists, try to download from huggingface
INFO:hy3dgen.shapgen:Model path not exists, try to download from huggingface


Loading input image: /content/Gemini_Generated_Image_vy9wrqvy9wrqvy9w.jpg


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

2026-11-03 05:45:21,018 - hy3dgen.shapgen - INFO - Loading model from /root/.cache/huggingface/hub/models--tencent--Hunyuan3D-2/snapshots/9cd649ba6913f7a852e3286bad86bfa9a2d83dcf/hunyuan3d-dit-v2-0/model.fp16.safetensors
INFO:hy3dgen.shapgen:Loading model from /root/.cache/huggingface/hub/models--tencent--Hunyuan3D-2/snapshots/9cd649ba6913f7a852e3286bad86bfa9a2d83dcf/hunyuan3d-dit-v2-0/model.fp16.safetensors


PointCrossAttentionEncoder INFO: pc_sharpedge_size is given, using pc_size=5120, pc_sharpedge_size=5120


In [ ]:
# @title 🧪 Run 3D Model Generation and STL Export test

import sys

try:
    # Using the defined workflow, let's trigger the generation and export process
    print("--- Starting Generation & Export Process ---")

    # Run synthesis using the uploaded image
    generated_obj_path = run_hunyuan3d_workflow(
        input_filename="/content/Gemini_Generated_Image_vy9wrqvy9wrqvy9w.jpg",
        output_filename="custom_model_raw.obj",
        project_folder_name="3D Print Projects"
    )

    # Import trimesh to convert to STL format
    import trimesh
    import os

    if os.path.exists(generated_obj_path):
        print("\n--- Converting OBJ to STL ---")
        # Load synthesized OBJ
        mesh = trimesh.load(generated_obj_path)

        # Define output path
        base, _ = os.path.splitext(generated_obj_path)
        stl_path = base + ".stl"

        # Export to STL
        mesh.export(stl_path)

        print(f"\n✅ Success! STL file saved to: {stl_path}")
        print(f"File size: {os.path.getsize(stl_path) / (1024 * 1024):.2f} MB")

        # Audit of the final STL file
        print("\n=== STL MESH AUDIT ===")
        print(f"Vertices: {len(mesh.vertices):,}")
        print(f"Faces:    {len(mesh.faces):,}")
        print(f"Is Watertight (Manifold): {mesh.is_watertight}")
        print(f"Bounding Box Dimensions (XYZ): {mesh.extents.round(2)}")
    else:
        print("[Error]: Could not locate the generated OBJ file.")

except NameError as ne:
    print(f"[Setup Error]: Please make sure to run the prior setup cells and define `run_hunyuan3d_workflow` first. Details: {ne}")
except Exception as e:
    print(f"[Execution Error]: {e}")


2026-11-03 05:45:57,458 - hy3dgen.shapgen - INFO - Try to load model from local path: /root/.cache/hy3dgen/tencent/Hunyuan3D-2/hunyuan3d-dit-v2-0
INFO:hy3dgen.shapgen:Try to load model from local path: /root/.cache/hy3dgen/tencent/Hunyuan3D-2/hunyuan3d-dit-v2-0
2026-11-03 05:45:57,459 - hy3dgen.shapgen - INFO - Model path not exists, try to download from huggingface
INFO:hy3dgen.shapgen:Model path not exists, try to download from huggingface


--- Starting Generation & Export Process ---
Loading input image: /content/Gemini_Generated_Image_vy9wrqvy9wrqvy9w.jpg
Initializing Hunyuan3D Pipeline on GPU...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

2026-11-03 05:45:57,768 - hy3dgen.shapgen - INFO - Loading model from /root/.cache/huggingface/hub/models--tencent--Hunyuan3D-2/snapshots/9cd649ba6913f7a852e3286bad86bfa9a2d83dcf/hunyuan3d-dit-v2-0/model.fp16.safetensors
INFO:hy3dgen.shapgen:Loading model from /root/.cache/huggingface/hub/models--tencent--Hunyuan3D-2/snapshots/9cd649ba6913f7a852e3286bad86bfa9a2d83dcf/hunyuan3d-dit-v2-0/model.fp16.safetensors


PointCrossAttentionEncoder INFO: pc_sharpedge_size is given, using pc_size=5120, pc_sharpedge_size=5120
[Execution Error]: Error(s) in loading state_dict for SingleImageEncoder:
	Missing key(s) in state_dict: "main_image_encoder.model.encoder.layer.0.attention.q_proj.weight", "main_image_encoder.model.encoder.layer.0.attention.q_proj.bias", "main_image_encoder.model.encoder.layer.0.attention.k_proj.weight", "main_image_encoder.model.encoder.layer.0.attention.k_proj.bias", "main_image_encoder.model.encoder.layer.0.attention.v_proj.weight", "main_image_encoder.model.encoder.layer.0.attention.v_proj.bias", "main_image_encoder.model.encoder.layer.0.attention.o_proj.weight", "main_image_encoder.model.encoder.layer.0.attention.o_proj.bias", "main_image_encoder.model.encoder.layer.0.mlp.gate_proj.weight", "main_image_encoder.model.encoder.layer.0.mlp.gate_proj.bias", "main_image_encoder.model.encoder.layer.0.mlp.up_proj.weight", "main_image_encoder.model.encoder.layer.0.mlp.up_proj.bias", "ma

In [ ]:
import os

target_dir = "/content/drive/MyDrive/3D Print Projects"
target_file = "custom_model_raw.obj"
full_path = os.path.join(target_dir, target_file)

print(f"Checking directory: {target_dir}")
if os.path.exists(target_dir):
    print("Directory exists. Contents:")
    files = os.listdir(target_dir)
    if not files:
        print(" (Directory is empty)")
    else:
        for f in files:
            print(f" - {f}")

    if target_file in files:
        print(f"\n✅ SUCCESS: '{target_file}' was found in the folder!")
    else:
        print(f"\n❌ ERROR: '{target_file}' was NOT found in the folder.")
else:
    print(f"\n❌ ERROR: Directory '{target_dir}' does NOT exist.")

In [ ]:
# @title 🛠️ Install SolidCraft & Python Mesh Repair Tools
import os
import subprocess

print("--- Setting up SolidCraft & Repair Dependencies ---")

# 1. Install trimesh & Rtree for mesh healing operations
!pip install -q trimesh rtree

# 2. Install Rust toolchain if not already present in Colab
if not os.path.exists(os.path.expanduser('~/.cargo/bin/cargo')):
    print("Installing Rust toolchain...")
    !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y

# Add cargo to PATH for the current session
os.environ["PATH"] += f":{os.path.expanduser('~')}/.cargo/bin"

# 3. Clone and build SolidCraft
solidcraft_dir = "/content/solidcraft"
if not os.path.exists(solidcraft_dir):
    print("Cloning SolidCraft repository...")
    !git clone https://github.com/tangency-labs/solidcraft.git {solidcraft_dir}

print("Building SolidCraft release binary...")
subprocess.run(["cargo", "build", "--release", "-p", "solidcraft-cli"], cwd=solidcraft_dir, check=True)
print("✅ SolidCraft and dependencies are fully ready!")

In [ ]:
# @title 🚀 Run Full Pipeline: Image ➔ Hunyuan3D ➔ Mesh Cleanup ➔ SolidCraft ➔ Slicer-Ready File
import os
import sys
import subprocess
import importlib

# 1. Automatically clone Hunyuan3D-2 repository if missing
repo_path = "/content/Hunyuan3D-2"
if not os.path.exists(repo_path):
    print("📥 Hunyuan3D-2 repository not found. Cloning repository dynamically...")
    subprocess.run(["git", "clone", "--recurse-submodules", "https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git", repo_path], check=True)

# Ensure the parent directory is in sys.path
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

# 2. Ensure trimesh and pymeshlab are installed in the active session
try:
    import trimesh
    import pymeshlab
except ModuleNotFoundError:
    print("📦 Required mesh libraries not found. Installing dependencies dynamically...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "trimesh", "rtree", "pymeshlab"], check=True)
    import trimesh
    import pymeshlab

# Forcefully install Hunyuan3D-2 dynamically via pip to resolve packaging metadata
try:
    importlib.invalidate_caches()
    import hy3dgen
    print("✅ 'hy3dgen' imported successfully!")
except ModuleNotFoundError:
    print("📦 'hy3dgen' not fully registered. Installing the package globally into runtime...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "/content/Hunyuan3D-2"], check=True)
    importlib.invalidate_caches()
    import hy3dgen
    print("✅ 'hy3dgen' successfully compiled and imported!")

import PIL.Image
import torch
from google.colab import files

# Define paths matching your project directory structure
project_dir = "/content/drive/MyDrive/3D Print Projects"
raw_obj_output = "custom_model_raw.obj"
clean_obj_output = "custom_model_clean.obj"
final_print_file = "custom_model_print_ready.3mf"

# CRITICAL: Point input_image directly to the attached image file
input_image_path = "/content/Gemini_Generated_Image_vy9wrqvy9wrqvy9w.jpg"
raw_obj_path = os.path.join(project_dir, raw_obj_output)
clean_obj_path = os.path.join(project_dir, clean_obj_output)
final_output_path = os.path.join(project_dir, final_print_file)

# Dynamic image safeguard
if not os.path.exists(input_image_path):
    print(f"❌ ERROR: Input image not found at '{input_image_path}'. please upload it again.")

# Automatically define workflow if not loaded to prevent NameError after runtime restarts
if 'run_hunyuan3d_workflow' not in globals():
    print("🔄 Re-defining 'run_hunyuan3d_workflow' dynamically...")
    def run_hunyuan3d_workflow(
        input_filename="input_model.png",
        output_filename="output_mesh.obj",
        project_folder_name="3D Print Projects",
        drive_base="/content/drive/MyDrive"
    ):
        project_dir_inner = os.path.join(drive_base, project_folder_name)
        if os.path.isabs(input_filename):
            input_image_path_inner = input_filename
        else:
            input_image_path_inner = os.path.join(project_dir_inner, input_filename)

        output_mesh_path_inner = os.path.join(project_dir_inner, output_filename)
        os.makedirs(project_dir_inner, exist_ok=True)
        if not hasattr(PIL.Image, "ANTIALIAS"):
            PIL.Image.ANTIALIAS = PIL.Image.LANCZOS
        repo_path_inner = "/content/Hunyuan3D-2"
        if repo_path_inner not in sys.path:
            sys.path.insert(0, repo_path_inner)
        from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline
        if not os.path.exists(input_image_path_inner):
            raise FileNotFoundError(f"Input image not found: {input_image_path_inner}")
        print(f"Loading input image: {input_image_path_inner}")
        pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
            "tencent/Hunyuan3D-2", torch_dtype=torch.float16, device="cuda"
        )
        print(f"Synthesizing 3D mesh for ‘{output_filename}’...")
        mesh = pipeline(image=input_image_path_inner)[0]
        mesh.export(output_mesh_path_inner)
        return output_mesh_path_inner

print("=== STEP 1: Running Hunyuan3D Synthesis ==")
try:
    generated_path = run_hunyuan3d_workflow(
        input_filename=input_image_path,
        output_filename=raw_obj_output,
        project_folder_name="3D Print Projects"
    )
except Exception as e:
    print(f"\n❌ ERROR during synthesis: {e}")
    generated_path = None

if generated_path:
    print("\n=== STEP 2: Pre-processing & Repairing Mesh for CAD/Printing ==")
    mesh = trimesh.load(generated_path, force='mesh')
    trimesh.repair.fill_holes(mesh)
    trimesh.repair.fix_normals(mesh)
    trimesh.repair.fix_inversion(mesh)

    if len(mesh.faces) > 50000:
        print(f"Decimating mesh faces from {len(mesh.faces):,} to 50,000...")
        mesh = mesh.simplify_quadric_decimation(face_count=50000)

    mesh.export(clean_obj_path)
    print(f"Cleaned mesh saved (Watertight: {mesh.is_watertight})")

    print("\n=== STEP 3: Processing through SolidCraft ==")
    os.environ["PATH"] += f":{os.path.expanduser('~')}/.cargo/bin"
    solidcraft_cli = "/content/solidcraft/target/release/solidcraft-cli"

    cmd = [
        solidcraft_cli,
        "run",
        "--input", clean_obj_path,
        "--out", final_output_path
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0 and os.path.exists(final_output_path):
        print(f"\n✅ Success! Print-ready file generated at: {final_output_path}")
        print(f"File size: {os.path.getsize(final_output_path) / (1024 * 1024):.2f} MB")

        print("\n=== STEP 4: Triggering Local Download ==")
        files.download(final_output_path)
    else:
        print("SolidCraft CLI execution notice/output:", result.stdout)
        if result.stderr:
            print("Stderr details:", result.stderr)
